In [1]:
import glob
import os
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.onnx as onnx
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

In [2]:
DATASET_DIR = "dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")
CHECKPOINT_PATH = "qrbitnet.pt"

QR_SIZE = 21

SPLIT_VALUE = 0.2
BATCH_SIZE = 64
EPOCHS = 5

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce GTX 1650


In [4]:
class QRCodeDataset(Dataset):
    def __init__(self, images_dir: str, labels_dir: str) -> None:
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images = sorted(glob.glob(os.path.join(self.images_dir, "*.png")))

        items = []
        for image_path in self.images:
            base = os.path.splitext(os.path.basename(image_path))[0]
            label_path = os.path.join(self.labels_dir, f"{base}.txt")
            items.append((image_path, label_path))
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def read_label(self, path: str) -> torch.Tensor:
        with open(path, "r", encoding="utf-8") as file:
            lines = [line.strip() for line in file.readlines() if line.strip()]

        matrix = []
        for line in lines:
            bits = line.split()
            row = [int(bit) for bit in bits]
            matrix.append(row)

        return torch.tensor(matrix, dtype=torch.float32)
        
    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor, str]:
        image_path, label_path = self.items[index]

        image = Image.open(image_path).convert("L")
        array = np.array(image, dtype=np.float32) / 255.0

        x = torch.from_numpy(array).unsqueeze(0)
        y = self.read_label(label_path)

        return x, y, os.path.basename(image_path)

In [5]:
dataset = QRCodeDataset(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR)
train_size = int((1 - SPLIT_VALUE) * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

In [6]:
class ResBlock(nn.Module):
    def __init__(self, channels: int, num_groups: int = 8):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups, channels)
        self.activation = nn.SiLU(inplace=True)

    def forward(self, x):
        h = self.activation(self.gn1(self.conv1(x)))
        h = self.gn2(self.conv2(h))
        return self.activation(x + h)


class QRBitNet(nn.Module):
    def __init__(self, width: int = 128, num_groups: int = 8, depth: int = 6):
        super().__init__()
        self.patch = nn.Conv2d(1, width, kernel_size=10, stride=10, bias=False)
        self.gn0 = nn.GroupNorm(num_groups, width)
        self.blocks = nn.Sequential(*[ResBlock(width, num_groups=num_groups) for _ in range(depth)])        
        self.head = nn.Conv2d(width, 1, kernel_size=1, bias=True)
        self.activation = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.activation(self.gn0(self.patch(x)))
        x = self.blocks(x)
        x = self.head(x).squeeze(1)
        return x

In [7]:
model = QRBitNet().to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

In [ ]:
@torch.no_grad()
def compute_metrics(
    logits: torch.Tensor,
    y_true: torch.Tensor,
) -> Tuple[float, float]:
    probabilities = torch.sigmoid(logits)
    y_pred = (probabilities > 0.5).to(torch.float32)

    bit_accuracy = (y_pred == y_true).to(torch.float32).mean().item()

    correct_samples = (y_pred == y_true).view(y_true.size(0), -1).all(dim=1)
    exact_accuracy = correct_samples.to(torch.float32).mean().item()

    return bit_accuracy, exact_accuracy


def train_one_epoch(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: torch.nn.Module,
    device: str,
    scaler: Optional[torch.cuda.amp.GradScaler] = None,
) -> float:
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Train", leave=False)
    
    for x, y, _ in progress_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optim.zero_grad(set_to_none=True)

        use_amp = (scaler is not None) and (device == "cuda")
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            logits = model(x)
            loss = loss_fn(logits, y)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
        else:
            loss.backward()
            optim.step()

        total_loss += loss.item() * x.size(0)
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_one_epoch(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    device: str,
) -> Tuple[float, float, float]:
    model.eval()
    total_loss = 0.0
    bit_accuracies, exact_accuracies = [], []
    progress_bar = tqdm(loader, desc="Val", leave=False)
    
    for x, y, _ in progress_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = loss_fn(logits, y)
        total_loss += loss.item() * x.size(0)

        bit_accuracy, exact_accuracy = compute_metrics(logits, y)
        bit_accuracies.append(bit_accuracy)
        exact_accuracies.append(exact_accuracy)

        progress_bar.set_postfix(bit_accuracy=f"{bit_accuracy:.4f}", exact_accuracy=f"{exact_accuracy:.4f}")

    return (
        total_loss / len(loader.dataset),
        float(np.mean(bit_accuracies)),
        float(np.mean(exact_accuracies)),
    )

In [10]:
best_bit_accuracy = -1.0
best_exact_accuracy = -1.0
best_epoch = -1
start_epoch = 1

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])

    best_bit_accuracy = checkpoint.get("best_bit_accuracy", -1.0)
    best_exact_accuracy = checkpoint.get("best_exact_accuracy", -1.0)
    best_epoch = checkpoint.get("best_epoch", -1)
    start_epoch = best_epoch + 1

    print(
        f"Loaded checkpoint from {CHECKPOINT_PATH} | "
        f"Best bit accuracy: {best_bit_accuracy:.4f} | "
        f"Best exact accuracy: {best_exact_accuracy:.4f} | "
        f"Best epoch: {best_epoch} | "
        f"Resuming at epoch {start_epoch}"
    )

In [11]:
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optim,
        loss_fn,
        DEVICE,
        scaler=scaler,
    )
    val_loss, val_bit_accuracy, val_exact_accuracy = eval_one_epoch(
        model,
        val_loader,
        loss_fn,
        DEVICE,
    )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Training loss: {train_loss:.4f} | "
        f"Validation loss: {val_loss:.4f} | "
        f"Validation bit accuracy: {val_bit_accuracy:.4f} | "
        f"Validation exact accuracy: {val_exact_accuracy:.4f}"
    )
    if val_bit_accuracy > best_bit_accuracy:
        best_bit_accuracy = val_bit_accuracy
        best_exact_accuracy = val_exact_accuracy
        best_epoch = epoch

        torch.save(
            {
                "model_state": model.state_dict(),
                "best_bit_accuracy": best_bit_accuracy,
                "best_exact_accuracy": best_exact_accuracy,
                "best_epoch": best_epoch,
            },
            CHECKPOINT_PATH,
        )

        print(
            f"New best model saved with validation bit accuracy: "
            f"{best_bit_accuracy:.4f}."
        )

print(
    f"\nTraining finished. The best validation bit accuracy was "
    f"{best_bit_accuracy:.4f}, achieved at epoch {best_epoch}.\n\n"
)

Epoch 1/5 | Training loss: 0.1889 | Validation loss: 0.0151 | Validation bit accuracy: 0.9951 | Validation exact accuracy: 0.6319
New best model saved with validation bit accuracy: 0.9951.


Epoch 2/5 | Training loss: 0.0048 | Validation loss: 0.0009 | Validation bit accuracy: 0.9998 | Validation exact accuracy: 0.9521
New best model saved with validation bit accuracy: 0.9998.


Epoch 3/5 | Training loss: 0.0006 | Validation loss: 0.0020 | Validation bit accuracy: 0.9994 | Validation exact accuracy: 0.8653


Epoch 4/5 | Training loss: 0.0008 | Validation loss: 0.0002 | Validation bit accuracy: 0.9999 | Validation exact accuracy: 0.9893
New best model saved with validation bit accuracy: 0.9999.


Epoch 5/5 | Training loss: 0.0001 | Validation loss: 0.0002 | Validation bit accuracy: 0.9999 | Validation exact accuracy: 0.9859

Training finished. The best validation bit accuracy was 0.9999, achieved at epoch 4.




In [15]:
class QRBitNetBits(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.base(x)
        probabilities = torch.sigmoid(logits)
        bits = (probabilities < 0.5).to(torch.float32)
        return bits

In [16]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()

export_model = QRBitNetBits(model).eval()
dummy = torch.randn(1, 1, 210, 210, device=DEVICE)
onnx.export(
    export_model,
    dummy,
    "qrbitnet.onnx",
    opset_version=18,
    dynamo=False
)